# 🎨 Anime NSFW Image Generator — v4
### Animagine XL 3.1 · No Login Required · Colab T4 GPU

---

## ✅ Run in this exact order:

| # | Cell | What happens |
|---|------|--------------|
| 1 | **GPU Check** | Confirms T4 is attached |
| 2 | **Install** | Installs only diffusers — then **auto-restarts kernel** |
| 3 | **Verify** | Run after restart — confirms everything imports correctly |
| 4 | **Load Model** | Downloads ~6.6 GB first time, then cached |
| 5 | **Launch UI** | Opens the image generator |

> ⚠️ After Cell 2 restarts the kernel, a **"Session crashed"** popup will appear — this is **completely normal and expected**. Just click OK, then continue from **Cell 3**. Do NOT re-run Cell 2.

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 of 5 — GPU Check                                    ║
# ║  Must show "Tesla T4" below.                                 ║
# ║  If it shows nothing: Runtime → Change runtime type →       ║
# ║  select T4 GPU → Save, then reconnect and try again.        ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0 and result.stdout.strip():
    gpu_info = result.stdout.strip()
    print(f'✅ GPU detected: {gpu_info}')
    print()
    print('Python version:', sys.version)
else:
    print('❌ No GPU found!')
    print('   Fix: Runtime → Change runtime type → T4 GPU → Save')
    raise SystemExit('Please enable T4 GPU and restart.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 of 5 — Install diffusers + Restart Kernel           ║
# ║                                                              ║
# ║  ONLY diffusers, accelerate, and safetensors are installed.  ║
# ║  torch, transformers, and gradio are already pre-installed   ║
# ║  in Colab at the correct versions — we do NOT touch them.    ║
# ║  Installing xformers is intentionally skipped — it breaks   ║
# ║  torchvision on Colab (operator nms does not exist error).  ║
# ║                                                              ║
# ║  A kernel restart is required after install so Python loads  ║
# ║  the new diffusers instead of any cached old version.        ║
# ║  The "Session crashed" popup is NORMAL — just click OK.      ║
# ║  Then continue from Cell 3. Do NOT re-run this cell.        ║
# ╚══════════════════════════════════════════════════════════════╝

print('📦 Installing diffusers, accelerate, safetensors...')
print('   (NOT touching torch / transformers / gradio — already correct in Colab)')
print('─' * 62)

import subprocess, sys

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + list(packages)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-500:] if r.stderr else '(none)')
    return r.returncode

codes = [
    pip_install('diffusers'),          # must be latest — fixes FLAX_WEIGHTS_NAME and shard_checkpoint errors
    pip_install('accelerate'),         # needed by diffusers pipeline
    pip_install('safetensors'),        # needed to load .safetensors model files
]

if any(c != 0 for c in codes):
    print('⚠️  One or more packages had install warnings above (usually harmless).')
else:
    print('✅ All packages installed successfully!')

print('─' * 62)
print('🔄 Restarting kernel to load the newly installed versions...')
print('   "Session crashed" popup is NORMAL — click OK.')
print('   Then run Cell 3 → Cell 4 → Cell 5.')
print('   Do NOT re-run this cell after restart.')

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 of 5 — Verify (run this AFTER the kernel restart)   ║
# ║  Checks that all imports work before loading the model.     ║
# ║  If you see any errors here, see the fix instructions.      ║
# ╚══════════════════════════════════════════════════════════════╝

import sys
print(f'Python: {sys.version}')
print()

# ── torch ──────────────────────────────────────────────────────
import torch
cuda_ok = torch.cuda.is_available()
print(f'torch        : {torch.__version__}')
print(f'CUDA available: {cuda_ok}')
if cuda_ok:
    print(f'GPU name     : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM         : {vram_gb:.1f} GB')
else:
    print('❌  CUDA not available — make sure T4 GPU is selected.')
    raise SystemExit('Need GPU.')
print()

# ── transformers ───────────────────────────────────────────────
import transformers
print(f'transformers : {transformers.__version__}')

# ── diffusers — the critical one ───────────────────────────────
import diffusers
print(f'diffusers    : {diffusers.__version__}')

# ── gradio ─────────────────────────────────────────────────────
import gradio
print(f'gradio       : {gradio.__version__}')
print()

# ── Test the exact import that was failing in previous versions ─
print('Testing pipeline import...')
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
print('✅ StableDiffusionXLPipeline  — OK')
print('✅ EulerAncestralDiscreteScheduler — OK')
print()
print('=' * 50)
print('✅ ALL CHECKS PASSED — safe to run Cell 4!')
print('=' * 50)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 of 5 — Load Model                                   ║
# ║  First run: downloads ~6.6 GB (5–10 min).                   ║
# ║  Later runs: loads from cache (~60 sec).                    ║
# ║  No HuggingFace account or login needed.                    ║
# ╚══════════════════════════════════════════════════════════════╝

import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

MODEL_ID = 'cagliostrolab/animagine-xl-3.1'

print(f'📥 Loading: {MODEL_ID}')
print('First run downloads ~6.6 GB — please wait patiently...')
print('─' * 62)

# Load in float16 to fit in T4 VRAM.
# NOTE: No variant='fp16' here — this model doesn't publish fp16 variant
# files, so using that argument would cause a 404 error.
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)

# Euler Ancestral sampler — official recommendation for Animagine XL 3.1
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipe.scheduler.config
)

# Move to GPU
pipe = pipe.to('cuda')

# Reduce VRAM usage — built-in, no extra packages needed
# Intentionally NOT using xformers: it conflicts with torchvision on Colab
pipe.enable_attention_slicing()

# SDXL does not have a safety_checker by design — skip that entirely

print('─' * 62)
print('✅ Model loaded!')
used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'🖥️  VRAM used : {used:.2f} GB')
print(f'🖥️  VRAM free : {total - used:.2f} GB')
print(f'🖥️  VRAM total: {total:.2f} GB')
print()
print('✅ Ready — run Cell 5 to open the image generator!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 of 5 — Image Generator UI                           ║
# ║  The interface will appear below this cell.                  ║
# ║  A public share link (72 hr) is also printed.               ║
# ╚══════════════════════════════════════════════════════════════╝

import gradio as gr
import torch, random, os
from datetime import datetime

LOCAL_DIR = '/content/generated_images'
os.makedirs(LOCAL_DIR, exist_ok=True)

RESOLUTIONS = {
    'Portrait   832 × 1216  📱  (recommended)': (832, 1216),
    'Square    1024 × 1024  ⬜'               : (1024, 1024),
    'Landscape 1216 × 832   🖼️'               : (1216, 832),
    'Tall       768 × 1344  📏'               : (768, 1344),
}

DEFAULT_POS = (
    'masterpiece, best quality, newest, absurdres, highres, '
    '1girl, solo, long hair, blue eyes, smile, looking at viewer, explicit'
)
DEFAULT_NEG = (
    'lowres, (bad), text, error, fewer, extra, missing, worst quality, '
    'jpeg artifacts, low quality, watermark, unfinished, displeasing, '
    'oldest, early, chromatic aberration, signature, extra digits, '
    'artistic error, username, scan, bad anatomy, bad hands'
)


def generate(pos, neg, res_label, steps, cfg, seed_val, save_drive,
             progress=gr.Progress()):
    seed = random.randint(0, 2**32 - 1) if int(seed_val) == -1 else int(seed_val)
    w, h = RESOLUTIONS[res_label]
    gen  = torch.Generator(device='cuda').manual_seed(seed)

    progress(0.05, desc='⏳ Starting...')

    # Step progress callback
    step_count = [0]
    def on_step(p, i, t, kwargs):
        step_count[0] = i + 1
        pct = 0.05 + 0.90 * ((i + 1) / int(steps))
        progress(min(pct, 0.95), desc=f'🔄 Step {i+1} / {int(steps)}')
        return kwargs

    try:
        out = pipe(
            prompt=pos,
            negative_prompt=neg,
            width=w,
            height=h,
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            generator=gen,
            callback_on_step_end=on_step,
        )
    except RuntimeError as e:
        torch.cuda.empty_cache()
        if 'out of memory' in str(e).lower():
            return (
                None,
                '❌ VRAM out of memory!',
                'Fix: lower Steps to 15, switch to Portrait resolution, '
                'then scroll down and run the VRAM Cleanup cell, and try again.'
            )
        return None, f'❌ Generation error: {e}', ''

    progress(1.0, desc='✅ Done!')
    img = out.images[0]

    # Save locally
    ts   = datetime.now().strftime('%Y%m%d_%H%M%S')
    name = f'anime_{ts}_seed{seed}.png'
    path = os.path.join(LOCAL_DIR, name)
    img.save(path)
    msg  = f'💾 Saved: {path}'

    # Optionally save to Google Drive
    drive_dir = '/content/drive/MyDrive/AI_Anime_Images'
    if save_drive:
        if os.path.exists('/content/drive/MyDrive'):
            os.makedirs(drive_dir, exist_ok=True)
            img.save(os.path.join(drive_dir, name))
            msg += f'\n📁 Saved to Drive: {drive_dir}/{name}'
        else:
            msg += '\n⚠️  Drive not mounted. Run: from google.colab import drive; drive.mount("/content/drive")'

    info = f'📐 {w}×{h}   🔢 Steps: {int(steps)}   ⚙️ CFG: {cfg}   🎲 Seed: {seed}'
    return img, info, msg


# ── Build UI ─────────────────────────────────────────────────────
CSS = """
.gen-btn {
    background: linear-gradient(135deg, #6d28d9, #db2777) !important;
    color: #fff !important;
    font-size: 17px !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 10px !important;
    min-height: 52px !important;
}
.gen-btn:hover { opacity: 0.88 !important; }
"""

with gr.Blocks(
    title='🎨 Anime AI Generator',
    theme=gr.themes.Soft(primary_hue='purple', secondary_hue='pink'),
    css=CSS
) as demo:

    gr.Markdown("""
    # 🎨 Anime NSFW Image Generator
    **Model:** Animagine XL 3.1 &nbsp;|&nbsp; **GPU:** T4 15 GB &nbsp;|&nbsp;
    **Speed:** ~60–90 sec/image &nbsp;|&nbsp; **No login required ✅**

    > Use **comma-separated Danbooru tags** (not sentences).
    > Always start with quality tags. Add `explicit` for adult content.
    """)

    with gr.Row():

        with gr.Column(scale=5):
            pos_box = gr.Textbox(
                label='✏️  Positive Prompt  (what you WANT)',
                value=DEFAULT_POS, lines=4,
            )
            neg_box = gr.Textbox(
                label='🚫  Negative Prompt  (what to AVOID)',
                value=DEFAULT_NEG, lines=3,
            )
            res_radio = gr.Radio(
                label='📐  Resolution',
                choices=list(RESOLUTIONS.keys()),
                value='Portrait   832 × 1216  📱  (recommended)',
            )
            with gr.Row():
                steps_sl = gr.Slider(label='🔢 Steps', minimum=10, maximum=35, step=1, value=25)
                cfg_sl   = gr.Slider(label='⚙️ CFG Scale', minimum=3.0, maximum=12.0, step=0.5, value=7.0)
            seed_box = gr.Number(label='🎲 Seed  (−1 = random)', value=-1, precision=0)
            drive_cb = gr.Checkbox(
                label='📁 Also save to Google Drive',
                value=False,
                info='Mount Drive first: from google.colab import drive; drive.mount("/content/drive")'
            )
            gen_btn = gr.Button('🚀  Generate Image', elem_classes='gen-btn', variant='primary')

            gr.Markdown('### 💡 Example prompts (click to load):')
            gr.Examples(
                label='',
                examples=[
                    ['masterpiece, best quality, newest, absurdres, 1girl, solo, long silver hair, violet eyes, magical girl outfit, wand, stars, night sky, sensitive', DEFAULT_NEG],
                    ['masterpiece, best quality, newest, absurdres, 1girl, solo, short red hair, green eyes, school uniform, classroom, sunlight, smile, rating:safe', DEFAULT_NEG],
                    ['masterpiece, best quality, newest, absurdres, 1girl, solo, long black hair, red eyes, kimono, cherry blossoms, night, lanterns, nsfw', DEFAULT_NEG],
                    ['masterpiece, best quality, newest, absurdres, 2girls, blue hair, pink hair, beach, summer, bikini, ocean, sunset, explicit', DEFAULT_NEG],
                    ['masterpiece, best quality, newest, absurdres, 1girl, elf ears, white hair, forest, fantasy armor, bow and arrow, detailed background, rating:safe', DEFAULT_NEG],
                ],
                inputs=[pos_box, neg_box],
            )

        with gr.Column(scale=5):
            out_img  = gr.Image(label='🖼️  Generated Image', type='pil', height=550)
            info_box = gr.Textbox(label='ℹ️  Generation Info', interactive=False, lines=1)
            save_box = gr.Textbox(label='💾  Save Status',     interactive=False, lines=2)

    gen_btn.click(
        fn=generate,
        inputs=[pos_box, neg_box, res_radio, steps_sl, cfg_sl, seed_box, drive_cb],
        outputs=[out_img, info_box, save_box],
    )

    # ── Prompting Guide ────────────────────────────────────────
    with gr.Accordion('📖  Prompting Guide & Tag Reference', open=False):
        gr.Markdown("""
        ## How to write prompts for Animagine XL 3.1

        Use **comma-separated Danbooru tags**, not sentences. Order matters — put important tags first.

        ### ⭐ Always start with quality tags:
        ```
        masterpiece, best quality, newest, absurdres, highres
        ```

        ### 👤 Subject:  `1girl`  `1boy`  `2girls`  `solo`  `multiple girls`

        ### 💇 Hair:  `long hair`  `short hair`  `twin tails`  `ponytail`  `braids`
        Color: `blonde hair`  `black hair`  `blue hair`  `white hair`  `silver hair`  `pink hair`

        ### 👁️ Eyes:  `blue eyes`  `red eyes`  `gold eyes`  `green eyes`  `heterochromia`

        ### 👗 Outfits:
        `school uniform`  `kimono`  `maid outfit`  `magical girl outfit`  `yukata`
        `bikini`  `lingerie`  `nude`  `naked`  `topless`  `swimsuit`

        ### 🏞️ Settings:
        `indoors`  `outdoors`  `bedroom`  `beach`  `onsen`  `cherry blossoms`
        `night sky`  `sunset`  `moonlight`  `forest`  `cityscape`

        ### 📷 Camera / framing:
        `close-up`  `upper body`  `full body`  `from above`  `from behind`  `looking at viewer`

        ### 🔞 Content level tags:
        | Tag | What it produces |
        |-----|------------------|
        | `rating:safe` | Fully safe for work |
        | `sensitive` | Suggestive, no nudity |
        | `nsfw` | Adult, partial nudity |
        | `explicit` | Fully explicit adult content |

        ### ⚙️ Settings guide:
        | Setting | Best value | Notes |
        |---------|------------|-------|
        | Steps | 20–28 | 25 is a good balance |
        | CFG Scale | 6–8 | 7 is the sweet spot — don't go above 10 |
        | Seed | -1 | Random each run; copy seed from Info box to reproduce |
        | Resolution | 832×1216 | Best for characters |

        **To reproduce an image:** Copy the seed number from the ℹ️ Info box,
        paste it into the Seed field, keep the same prompt and settings.
        """)

    # ── VRAM Cleanup (inline, no separate cell needed) ──────────
    with gr.Accordion('🧹  VRAM Cleanup  (use if you get Out of Memory errors)', open=False):
        gr.Markdown("""
        If you get an **Out of Memory** error, run this in a new Colab cell:
        ```python
        import torch, gc
        del pipe
        gc.collect()
        torch.cuda.empty_cache()
        print('VRAM cleared — re-run Cell 4 to reload the model')
        ```
        """)

    # ── Download images ─────────────────────────────────────────
    with gr.Accordion('⬇️  Download All Images as ZIP', open=False):
        gr.Markdown("""
        Run this in a new Colab cell to download all generated images as a zip:
        ```python
        import zipfile, os
        from google.colab import files
        DIR = '/content/generated_images'
        imgs = [f for f in os.listdir(DIR) if f.endswith('.png')]
        if imgs:
            with zipfile.ZipFile('/content/anime_images.zip', 'w') as zf:
                for f in imgs: zf.write(f'{DIR}/{f}', f)
            files.download('/content/anime_images.zip')
        else:
            print('No images yet!')
        ```
        """)

print('🚀 Launching...')
demo.launch(share=True, debug=False)